# Topic Modeling (LDA) - Komentar Threads Karhutla
**Anggota 5 - Topic Modeling Analyst**

Notebook ini melatih model LDA dan menyimpan hasilnya ke `data/model_output/` untuk dibaca dashboard
(`pages/4_Topic_Analysis.py` -> `src/topic_modeling.py`). Dashboard **tidak** melatih ulang model.

Notebook harus bisa dijalankan ulang dari sel paling atas (Run All). Lama jalan sekitar 5-15 menit (sel pemilihan K).

## 1. Import library

In [ ]:
import os

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

nltk.download("stopwords", quiet=True)

## 2. Load dataset
Data dari Anggota 2: `data/processed/comments_clean.csv` (11.017 komentar; kolom sudah sesuai kontrak).
Kolom yang dipakai: `comment_id`, `text_clean` (teks bersih untuk LDA), `comment_text` (teks asli, hanya untuk contoh komentar).
Bila file/kolom berganti, cukup ubah **sel konfigurasi ini**.

In [ ]:
# ---- KONFIGURASI (ubah di sini saja) ----
FILE_DATA = "../data/processed/comments_clean.csv"
KOLOM_ID = "comment_id"
KOLOM_TEKS = "text_clean"
KOLOM_ASLI = "comment_text"
OUTPUT_DIR = "../data/model_output"
K_RANGE = range(3, 9)              # jumlah topik yang dibandingkan lewat coherence
RANDOM_STATE = 42

df = pd.read_csv(FILE_DATA)
df[KOLOM_TEKS] = df[KOLOM_TEKS].fillna("")   # text_clean kosong (sekitar 34 baris) jangan jadi kata "nan"
print(df.shape)
df[[KOLOM_ID, KOLOM_TEKS]].head()

## 3. Data preparation
Tokenisasi untuk LDA: pecah kalimat jadi kata, buang stopword (nltk Indonesia + Inggris + kata gaul/filler),
kata < 3 huruf, dan angka. Komentar dengan kurang dari 3 kata bermakna dibuang.

In [ ]:
stop = set(stopwords.words("indonesian")) | set(stopwords.words("english"))
stop |= {"yg", "gak", "ga", "nggak", "tp", "dgn", "aja", "sih", "deh", "kok",
         "udah", "udh", "nya", "jd", "utk", "krn", "dr", "sm", "bgt", "banget",
         "emang", "kan", "lah", "mah", "jg", "juga", "lg", "lagi", "gue", "gua",
         "lu", "lo", "aku", "kita", "kami", "kalo", "kalau", "biar", "tuh", "nih",
         "kak", "kakak", "yaa", "yaaa", "sampe", "amin", "aamiin", "wkwk", "wkwkw",
         "wkwkwk", "gk", "gtu", "gitu", "gini", "bener2", "orng", "tau"}

def tokenisasi(teks):
    return [t for t in str(teks).split() if t not in stop and len(t) > 2 and not t.isdigit()]

df["token"] = df[KOLOM_TEKS].map(tokenisasi)
df = df[df["token"].str.len() >= 3].reset_index(drop=True)
print(len(df), "komentar setelah tokenisasi")

In [ ]:
# Kamus kata + corpus (format yang dibaca gensim)
kamus = Dictionary(df["token"])
kamus.filter_extremes(no_below=5, no_above=0.5)   # buang kata terlalu langka / terlalu umum
korpus = [kamus.doc2bow(t) for t in df["token"]]

keep = [i for i, b in enumerate(korpus) if len(b) >= 3]
df = df.iloc[keep].reset_index(drop=True)
korpus = [korpus[i] for i in keep]
print(len(kamus), "kata di kamus,", len(df), "komentar")

## 4. Proses analisis - memilih jumlah topik
Melatih LDA untuk beberapa jumlah topik dan membandingkan coherence (c_v). Hasil dengan data final (11.017 komentar, 7.769 setelah filter):
K=3 **0.4242**, K=4 0.4163, K=5 0.4149, K=6 0.3991, K=7 0.4237, K=8 0.3710.


In [ ]:
hasil = []
for k in K_RANGE:
    m = LdaModel(korpus, num_topics=k, id2word=kamus, passes=10, random_state=RANDOM_STATE)
    c = CoherenceModel(model=m, texts=df["token"], dictionary=kamus,
                       coherence="c_v", processes=1).get_coherence()
    hasil.append({"n_topics": k, "coherence_score": c})
    print(k, round(c, 4))
pd.DataFrame(hasil)

## 5. Hasil analisis
Jumlah topik dipilih **K = 4**: meskipun K=3 memiliki coherence sedikit lebih tinggi (0.4242 vs 0.4163), K=4 dipilih karena isi topiknya lebih terbaca dan lebih bermakna secara interpretasi. Selisih coherence antar K tipis, sehingga pertimbangan keterbacaan menjadi penentu.


In [ ]:
K = 4
model = LdaModel(korpus, num_topics=K, id2word=kamus, passes=10, random_state=RANDOM_STATE)

# topik dominan tiap komentar
dominan = []
for bow in korpus:
    dist = model.get_document_topics(bow, minimum_probability=0.0)
    tid, p = max(dist, key=lambda x: x[1])
    dominan.append((tid, p))
df["topic_id"] = [d[0] for d in dominan]
df["topic_probability"] = [d[1] for d in dominan]

for i in range(K):
    sub = df[df["topic_id"] == i]
    print(f"\n=== Topik {i}: {', '.join(w for w, _ in model.show_topic(i, topn=8))}")
    print("Jumlah komentar:", len(sub))
    for teks in sub.sample(min(3, len(sub)), random_state=1)[KOLOM_ASLI]:
        print("  -", str(teks)[:150])

Nama topik ditentukan dari kata dominan dan contoh komentar. **Cek urutan ID di atas** sebelum menjalankan sel export:
jika kata-kata tiap topik berbeda dari yang tertulis di komentar `NAMA_TOPIK`, sesuaikan nomornya.

In [ ]:
NAMA_TOPIK = {   # 0 indonesia/pemerintah/negara/perusahaan, 1 asap/kebakaran/lahan/api, 2 sawit/bencana/bantu/lahan, 3 allah/semoga/dibakar
    0: "Kritik ke Pemerintah dan Perusahaan",
    1: "Kondisi Kebakaran dan Lapangan",
    2: "Dampak Bencana dan Peran Sawit",
    3: "Doa dan Kecaman Pembakaran",
}
assert set(NAMA_TOPIK) == set(range(K)), "NAMA_TOPIK harus punya satu nama untuk tiap topic_id"
df["topic_name"] = df["topic_id"].map(NAMA_TOPIK)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. topic_assignments.csv  (comment_id, topic_id, topic_name, topic_probability)
(df[[KOLOM_ID, "topic_id", "topic_name", "topic_probability"]]
   .rename(columns={KOLOM_ID: "comment_id"})
   .to_csv(f"{OUTPUT_DIR}/topic_assignments.csv", index=False))

# 2. topic_words.csv  (topic_id, word, weight, rank)
baris = []
for i in range(K):
    for r, (wid, p) in enumerate(model.get_topic_terms(i, topn=30), start=1):
        baris.append({"topic_id": i, "word": kamus[wid], "weight": p, "rank": r})
pd.DataFrame(baris).to_csv(f"{OUTPUT_DIR}/topic_words.csv", index=False)

# 3. topic_metrics.csv  (model_name, n_topics, coherence_score, perplexity, model_config)
m = pd.DataFrame(hasil)
m.insert(0, "model_name", "LDA")
m["perplexity"] = np.nan
m["model_config"] = "passes=10; alpha=symmetric; random_state=42; no_below=5; no_above=0.5"
m.to_csv(f"{OUTPUT_DIR}/topic_metrics.csv", index=False)

# 4. topic_comparison.csv  (topic_id, lambda, word, weight, rank)
#    weight = relevansi = lambda*log P(w|topik) + (1-lambda)*log(P(w|topik)/P(w)), dihitung dari seluruh kosakata
total = np.zeros(len(kamus))
for bow in korpus:
    for wid, c in bow:
        total[wid] += c
p_korpus = total / total.sum()
tp = model.get_topics()   # matriks P(kata | topik)

baris = []
for lam in np.round(np.arange(0, 1.01, 0.1), 1):
    for i in range(K):
        rel = lam * np.log(tp[i]) + (1 - lam) * np.log(tp[i] / p_korpus)
        for r, wid in enumerate(np.argsort(-rel)[:20], start=1):
            baris.append({"topic_id": i, "lambda": lam, "word": kamus[wid],
                          "weight": rel[wid], "rank": r})
pd.DataFrame(baris).to_csv(f"{OUTPUT_DIR}/topic_comparison.csv", index=False)
print("selesai: 4 file tersimpan di", OUTPUT_DIR)

## 6. Kesimpulan dan interpretasi
| ID | Nama topik | Kata dominan |
|---|---|---|
| 0 | Kritik ke Pemerintah dan Perusahaan | indonesia, pemerintah, negara, rakyat, malaysia, perusahaan |
| 1 | Kondisi Kebakaran dan Lapangan | asap, kebakaran, lahan, kalimantan, api, hutan |
| 2 | Dampak Bencana dan Peran Sawit | sawit, bencana, air, bantu, lahan, pemerintah |
| 3 | Doa dan Kecaman Pembakaran | allah, semoga, bakar, dibakar, kalimantan, papua |

**Catatan metode dan keterbatasan**
- K=3 menghasilkan coherence sedikit lebih tinggi (0.4242 vs K=4: 0.4163), namun selisihnya tipis dan K=4 dipilih karena topiknya lebih terbaca.
- Topik 2 (Dampak Bencana dan Peran Sawit) dan Topik 3 (Doa dan Kecaman Pembakaran) agak campur aduk; beberapa komentar berisi elemen dari dua topik sekaligus. Ini wajar untuk komentar media sosial yang pendek dan beragam.
- Coherence c_v secara keseluruhan sedang (sekitar 0.42); wajar untuk komentar pendek.
- 183 komentar (sekitar 2%) bukan teks unik (copy-paste); dibiarkan karena kecil dan tidak menggeser topik.
- Data tidak difilter dengan `ada_kata_kunci` (terlalu ketat; banyak komentar relevan tidak memuat kata kuncinya).
- Setiap komentar dimasukkan ke satu topik dominan; `topic_probability` menunjukkan keyakinannya.
